<a href="https://colab.research.google.com/github/tuttlelm/nmr_tools/blob/main/BMRB_CHSQC_avg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np, pandas as pd
import itertools
import plotly.express as px

#download the cs_stat_aa_full.csv file from https://bmrb.io/ref_info/csstats.php?restype=aa&set=full&output=csv
#and upload here (click on files folder icon on left panel, there is an upload option there)

In [ ]:
#may be some atom names mismatched still between this dictionary and the BMRB stats atom names
ch_atoms = {'ALA': [('HA', 'CA'), ('HB*', 'CB')],
                'ARG': [('HA', 'CA'), ('HB*', 'CB'), ('HG*', 'CG'), ('HD*', 'CD')],
                'ASN': [('HA', 'CA'), ('HB*', 'CB')],
                'ASP': [('HA', 'CA'), ('HB*', 'CB')],
                'CYS': [('HA', 'CA'), ('HB*', 'CB')],
                'GLN': [('HA', 'CA'), ('HB*', 'CB'), ('HG*', 'CG')],
                'GLU': [('HA', 'CA'), ('HB*', 'CB'), ('HG*', 'CG')],
                'GLY': [('HA*', 'CA')],
                'HIS': [('HA', 'CA'), ('HB*', 'CB'), ('HD2', 'CD2'), ('HE1', 'CE1')],
                'ILE': [('HA', 'CA'), ('HB', 'CB'), ('HG2*', 'CG2'), ('HG1*', 'CG1'), ('HD*', 'CD1')],
                'LEU': [('HA', 'CA'), ('HB*', 'CB'), ('HG', 'CG'), ('HD1*', 'CD1'), ('HD2*', 'CD2')],
                'LYS': [('HA', 'CA'), ('HB*', 'CB'), ('HG*', 'CG'), ('HD*', 'CD'), ('HE*', 'CE')],
                'MET': [('HA', 'CA'), ('HB*', 'CB'), ('HG*', 'CG'), ('HE*', 'CE')],
                'PHE': [('HA', 'CA'), ('HB*', 'CB'), ('HD1', 'CD1'), ('HE1', 'CE1'), ('HD2', 'CD2'), ('HE2', 'CE2'),
                        ('HZ', 'CZ')],
                'PRO': [('HA', 'CA'), ('HB*', 'CB'), ('HG*', 'CG'), ('HD*', 'CD')],
                'SER': [('HA', 'CA'), ('HB*', 'CB')],
                'TRP': [('HA', 'CA'), ('HB*', 'CB'), ('HD1', 'CD1'), ('HE3', 'CE3'), ('HZ2', 'CZ2'),
                        ('HH2', 'CH2')],
                'TYR': [('HA', 'CA'), ('HB*', 'CB'), ('HD1', 'CD1'), ('HE1', 'CE1'), ('HD2', 'CD2'),
                        ('HE2', 'CE2')],
                'THR': [('HA', 'CA'), ('HB', 'CB'), ('HG', 'CG2')],
                'VAL': [('HA', 'CA'), ('HB', 'CB'), ('HG1*', 'CG1'), ('HG2*', 'CG2')],
                }

bmrb_cs_stats = pd.read_csv('cs_stat_aa_full.csv')
bmrb_cs_stats['orig_atom_id'] = bmrb_cs_stats['atom_id']
bmrb_cs_stats['atom_id'] = bmrb_cs_stats['orig_atom_id'].str.replace('M','H')
bmrb_cs_stats = bmrb_cs_stats.drop(bmrb_cs_stats[(bmrb_cs_stats['comp_id']=='THR') & (bmrb_cs_stats['atom_id']=='HG1')].index) #dumb remove of HG1


shifts_df = pd.DataFrame()
for res in bmrb_cs_stats['comp_id'].unique():
    for pair in ch_atoms[res]:

        x_atoms = bmrb_cs_stats[(bmrb_cs_stats['comp_id']==res) & (bmrb_cs_stats['atom_id'].str.startswith(pair[0].strip('*')))]['atom_id'].unique()
        y_atoms = bmrb_cs_stats[(bmrb_cs_stats['comp_id']==res) & (bmrb_cs_stats['atom_id'].str.startswith(pair[1].strip('*')))]['atom_id'].unique()

        if (len(x_atoms) == 0) | (len(y_atoms) == 0): continue

        sub_pairs = list(itertools.product(x_atoms,y_atoms))
        for p in sub_pairs:
            shift_df = pd.DataFrame()
            x = bmrb_cs_stats[(bmrb_cs_stats['comp_id']==res) & (bmrb_cs_stats['atom_id']==p[0])]['avg'].values
            xstd = bmrb_cs_stats[(bmrb_cs_stats['comp_id']==res) & (bmrb_cs_stats['atom_id']==p[0])]['std'].values
            y = bmrb_cs_stats[(bmrb_cs_stats['comp_id']==res) & (bmrb_cs_stats['atom_id']==p[1])]['avg'].values
            ystd = bmrb_cs_stats[(bmrb_cs_stats['comp_id']==res) & (bmrb_cs_stats['atom_id']==p[1])]['std'].values
            shift_df['Residue'] = [res]
            shift_df['x_atom'] = [p[0]]
            shift_df['y_atom'] = [p[1]]
            shift_df['x_shift'] = x
            shift_df['x_shift_std'] = xstd*0.674 #50% conf interval
            shift_df['y_shift'] = y
            shift_df['y_shift_std'] = ystd*0.674

            shifts_df = pd.concat([shifts_df,shift_df],ignore_index=True)


shifts_df['label'] = shifts_df['Residue'] + ' { ' + shifts_df['x_atom'] +', '+ shifts_df['y_atom'] +' }'
#display(shifts_df)
fig = px.scatter(shifts_df,x='x_shift',y='y_shift',error_x='x_shift_std',error_y='y_shift_std', hover_name='label',
            color="Residue", width = 1000, height = 600,
            labels={"color":"Residue",
                    #"symbol":"sym",
                "x_shift": '<sup>1</sup>H (ppm)',
                "y_shift": '<sup>13</sup>C (ppm)'})#.update(layout=dict(title=dict(x=0.5)))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(autorange="reversed")
fig.show()